In [2]:
!pip install ultralytics torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.2 MB/s eta 0:00:0000:010:01


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())
!nvidia-smi

True
Tesla T4
Sun May 24 20:02:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------

In [5]:
import os
import shutil
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path
from ultralytics import YOLO

def get_cluttered_images(xml_path):
    """
    Parses the annotations.xml to find images tagged as 'cluttered'.
    Returns a set of image filenames.
    """
    if not xml_path.exists():
        print(f"Warning: {xml_path} not found. No images will be marked as cluttered.")
        return set()
    tree = ET.parse(xml_path)
    root = tree.getroot()
    cluttered_images = set()

    for image in root.findall('image'):
        image_name = image.get('name')
        for tag in image.findall('tag'):
            if tag.get('label') == 'cluttered':
                cluttered_images.add(image_name)
                break
    return cluttered_images

def preprocess_dataset(input_dir, output_dir, target_class_name="graffiti", dry_run=False):
    """
    Preprocesses the dataset:
    - Merges all labels into one (target_class_name).
    - Filters out images without bounding boxes.
    - Uses annotations.xml to separate 'cluttered' images for evaluation.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    xml_path = input_path / "annotations.xml"

    # Train split (clean images)
    train_img_out = output_path / "images" / "train"
    train_lb_out = output_path / "labels" / "train"
    # Val split (cluttered images)
    val_img_out = output_path / "images" / "val"
    val_lb_out = output_path / "labels" / "val"

    if not dry_run:
        for d in [train_img_out, train_lb_out, val_img_out, val_lb_out]:
            d.mkdir(parents=True, exist_ok=True)

    cluttered_images = get_cluttered_images(xml_path)
    if dry_run:
        print(f"[Dry Run] Found {len(cluttered_images)} cluttered images in XML.")
    else:
        print(f"Found {len(cluttered_images)} cluttered images in XML.")

    label_files = list((input_path / "labels" / "train").glob("*.txt"))
    train_count = 0
    val_count = 0
    skipped_empty = 0
    skipped_no_img = 0

    for lb_file in label_files:
        img_name_stem = lb_file.stem

        # Find corresponding image
        img_file = None
        img_full_name = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
            candidate = input_path / "images" / "train" / f"{img_name_stem}{ext}"
            if candidate.exists():
                img_file = candidate
                img_full_name = f"{img_name_stem}{ext}"
                break

        if not img_file:
            skipped_no_img += 1
            continue

        with open(lb_file, 'r') as f:
            lines = f.readlines()

        if not lines:
            skipped_empty += 1
            continue

        new_labels = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            new_labels.append(f"0 {' '.join(parts[1:])}")

        # Decide split
        if img_full_name in cluttered_images:
            img_dest = val_img_out
            lb_dest = val_lb_out
            val_count += 1
        else:
            img_dest = train_img_out
            lb_dest = train_lb_out
            train_count += 1

        if not dry_run:
            shutil.copy(img_file, img_dest / img_file.name)
            with open(lb_dest / f"{img_name_stem}.txt", 'w') as f:
                f.write('\n'.join(new_labels) + '\n')

    print(f"Preprocessing results {'(Dry Run)' if dry_run else ''}:")
    print(f"- Training images (clean): {train_count}")
    print(f"- Validation images (cluttered): {val_count}")
    print(f"- Skipped empty: {skipped_empty}")
    print(f"- Skipped (no image found): {skipped_no_img}")

    if dry_run:
        return None

    # Create new data.yaml
    new_data_config = {
        'path': str(output_path.absolute()),
        'train': 'images/train',
        'val': 'images/val',
        'names': {0: target_class_name}
    }

    with open(output_path / "data.yaml", 'w') as f:
        yaml.dump(new_data_config, f)

    return output_path / "data.yaml"

def train_yolo(data_yaml, model_name="yolov8n.pt", epochs=50, imgsz=640, batch=16, device=None, **kwargs):
    if data_yaml is None:
        print("[Dry Run] Skipping training.")
        return None
    model = YOLO(model_name)
    results = model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        plots=True,
        single_cls=True,
        **kwargs
    )
    return results


def train_yolo_two_stage(
    data_yaml,
    model_name="yolov8n.pt",
    epochs=100,
    freeze_epochs=20,
    freeze_layers=10,
    imgsz=640,
    batch=16,
    device=None,
    stage2_lr_scale=0.1,
    run_name="train",
    **kwargs,
):
    """Two-stage fine-tune.

    Stage 1: backbone frozen (`freeze=freeze_layers`) for `freeze_epochs`.
    Stage 2: full network unfrozen for the remaining epochs with lr0
    scaled by `stage2_lr_scale` and warmup disabled.
    """
    if data_yaml is None:
        print("[Dry Run] Skipping training.")
        return None

    stage2_epochs = epochs - freeze_epochs
    if stage2_epochs <= 0:
        raise ValueError("epochs must exceed freeze_epochs")

    lr0 = kwargs.pop("lr0", 0.001)
    close_mosaic = kwargs.pop("close_mosaic", 10)
    warmup_epochs = kwargs.pop("warmup_epochs", 3)

    common = dict(
        data=str(data_yaml),
        imgsz=imgsz,
        batch=batch,
        device=device,
        plots=True,
        single_cls=True,
    )

    # Stage 1: frozen backbone, mosaic on through the whole stage
    stage1_model = YOLO(model_name)
    stage1_model.train(
        epochs=freeze_epochs,
        freeze=freeze_layers,
        lr0=lr0,
        warmup_epochs=warmup_epochs,
        close_mosaic=0,
        name=f"{run_name}-stage1",
        **common,
        **kwargs,
    )

    stage1_best = Path(stage1_model.trainer.save_dir) / "weights" / "best.pt"
    print(f"Stage 1 finished. Loading {stage1_best} for stage 2.")

    # Stage 2: unfreeze everything, smaller lr, no warmup
    stage2_model = YOLO(str(stage1_best))
    results = stage2_model.train(
        epochs=stage2_epochs,
        freeze=None,
        lr0=lr0 * stage2_lr_scale,
        warmup_epochs=0,
        close_mosaic=min(close_mosaic, stage2_epochs),
        name=f"{run_name}-stage2",
        **common,
        **kwargs,
    )
    return results


In [ ]:
DATASET_PATH = "/home/tino/Desktop/uni/S10/tfm/data/detect/merged"
preprocess_dataset(DATASET_PATH, "/home/tino/Desktop/uni/S10/tfm/data/detect/preprocessed")

In [9]:
DATASET_PATH = "/kaggle/input/datasets/luengor/graffiti-annotated-2"
SAVE_DIR = "/kaggle/working/yolo"
os.makedirs(SAVE_DIR, exist_ok=True)

IMAGE_SIZE = 960
BATCH_SIZE = 16

AUGMENT = {
    # Color
    'hsv_h': 0.015,
    'hsv_s': 0.5,
    'hsv_v': 0.5,
    'bgr': 0,

    # Geometric
    'degrees': 7.0,
    'translate': 0.1,
    'scale': 0.5,
    'shear': 0.0,
    'perspective': 0.0001,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.1
}

EPOCHS = 100
MODEL_NAME = "yolo11m.pt"

TRAINING = {
    'lr0': 0.001,
    'cos_lr': True,

    'dropout': 0.1,
    'multi_scale': 0.2,
    'cache': True,
    'warmup_epochs': 5,
    'close_mosaic': 20,
    'patience': 20,
    'optimizer': 'AdamW',
}

LOSSES = {
    'box': 8,
    'cls': 0.4,
    'dfl': 1.0,
}

FREEZE_EPOCHS = 20
FREEZE_LAYERS = 10
STAGE2_LR_SCALE = 0.1

train_yolo_two_stage(
    DATASET_PATH,
    model_name=MODEL_NAME,
    epochs=EPOCHS,
    freeze_epochs=FREEZE_EPOCHS,
    freeze_layers=FREEZE_LAYERS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=[0, 1],
    stage2_lr_scale=STAGE2_LR_SCALE,
    run_name="train-medium",
    **AUGMENT,
    **TRAINING,
    **LOSSES,
)

Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0, box=8, cache=True, cfg=None, classes=None, close_mosaic=0, cls=0.4, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/input/datasets/luengor/graffiti-annotated-2, degrees=7.0, deterministic=True, device=0,1, dfl=1.0, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.5, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.2, name=train-

In [7]:
def zip_run_folder(run_name: str, runs_dir: str = "/kaggle/working/runs/detect") -> str:
    """Zip a single Ultralytics run directory.

    `run_name` is the folder under `runs_dir` (e.g. "train-stage2", "train-stage2-3").
    Returns the zip path.
    """
    run_path = Path(runs_dir) / run_name
    if not run_path.is_dir():
        raise FileNotFoundError(run_path)
    archive_base = str(run_path)
    print(archive_base)
    shutil.make_archive(archive_base, "zip", archive_base)
    return f"{archive_base}.zip"


def zip_two_stage_run(run_name: str = "train", runs_dir: str = "/kaggle/working/runs/detect") -> list:
    """Zip both stages of a two-stage run. Picks the latest auto-suffixed dirs."""
    base = Path(runs_dir)
    zips = []
    for stage in ("stage1", "stage2"):
        candidates = sorted(base.glob(f"{run_name}-{stage}*"), key=lambda p: p.stat().st_mtime)
        if not candidates:
            print(f"No directory found for {run_name}-{stage}")
            continue
        zips.append(zip_run_folder(candidates[-1].name, runs_dir))
    return zips


In [10]:
zip_two_stage_run("train-medium")

/kaggle/working/runs/detect/train-medium-stage1
/kaggle/working/runs/detect/train-medium-stage2


['/kaggle/working/runs/detect/train-medium-stage1.zip',
 '/kaggle/working/runs/detect/train-medium-stage2.zip']